# Sistema Multiagente basado en Aprendizaje por Refuerzo para la Planificación y Monitoreo de Rutas Operativas en el Transporte de Carga

**Corredor Norte del Perú: Lima → Trujillo → Chiclayo → Piura → Sullana → Chulucanas**

---

**Universidad Nacional de Ingeniería — FIIS, Grupo 8**  
**Trabajo de Investigación II**  
**Noviembre 2026**

---

## Descripción

Este notebook propone el flujo inicial del trabajo de investigación. Es un **avance** que muestra cómo se vería el pipeline completo sin necesidad de ejecutar el preprocesamiento desde cero. Las celdas de código están diseñadas para cargar los datos ya preprocesados (red vial construida desde OpenStreetMap) y mostrar los componentes clave del sistema.

### Componentes del proyecto

| Componente | Estado | Descripción |
|---|---|---|
| Red vial (OSM) | ✅ Completado | 18 167 nodos, 28 642 tramos, 14 017.1 km |
| Benchmark (viajes reales) | ✅ Completado | 500/500 viajes reconstruidos |
| Simulador Dec-POMDP | ✅ Completado | 4 acciones, 3 agentes, eventos dinámicos |
| Algoritmo MARL | ✅ Implementado | QMIX (principal) + MAPPO (contraste) |
| Entrenamiento | ⏳ Pendiente | Código verificado, falta ejecución completa |
| Evaluación estadística | ⏳ Pendiente | Mann-Whitney vs. 3 baselines |

## 1. Configuración e importaciones

In [ ]:
# Importaciones estándar
import os
import sys
import json
import yaml
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib

matplotlib.rcParams['figure.figsize'] = (12, 6)
matplotlib.rcParams['font.size'] = 11

# Configurar ruta del proyecto
PROJECT_ROOT = os.path.dirname(os.path.abspath('__file__')) if '__file__' in dir() else os.getcwd()
sys.path.insert(0, PROJECT_ROOT)

# Rutas de datos
DATA_PROCESSED = os.path.join(PROJECT_ROOT, 'data', 'processed')
RED_VIAL_DIR = os.path.join(DATA_PROCESSED, 'red_vial')
BENCHMARK_DIR = os.path.join(DATA_PROCESSED, 'benchmark')
VALHALLA_DIR = os.path.join(DATA_PROCESSED, 'valhalla')
ARTIFACTS_DIR = os.path.join(PROJECT_ROOT, 'artifacts')

print(f'Proyecto: {PROJECT_ROOT}')
print(f'Datos procesados existen: {os.path.exists(DATA_PROCESSED)}')
print(f'Red vial existe: {os.path.exists(RED_VIAL_DIR)}')

## 2. Cargar la configuración del escenario

El escenario se define en `config/escenario_norte.yaml` y describe el corredor logístico, las ciudades, la flota, los costos operativos y los eventos dinámicos calibrados con SUTRAN/MTC.

In [ ]:
# Cargar configuración del escenario
config_path = os.path.join(PROJECT_ROOT, 'config', 'escenario_norte.yaml')

if os.path.exists(config_path):
    with open(config_path, 'r', encoding='utf-8') as f:
        config = yaml.safe_load(f)
    
    print('=== Escenario operativo ===')
    print(f'Nombre: {config["nombre"]}')
    print(f'Descripción: {config["descripcion"].strip()}')
    print(f'\nCiudades del corredor:')
    for ciudad in config['ciudades']:
        print(f'  {ciudad["id"]:3s} — {ciudad["nombre"]:15s} | rol: {ciudad["rol"]:20s} | distribuidores: {ciudad["distribuidores"]}')
    
    print(f'\nFlota: {config["flota"]["n_vehiculos"]} vehículos')
    for tipo in config['flota']['tipos']:
        print(f'  {tipo["id"]:8s} — {tipo["nombre"]:25s} | cap: {tipo["capacidad_t"]}t | vel: {tipo["velocidad_max_kmh"]} km/h')
    
    print(f'\nEventos dinámicos (calibración SUTRAN):')
    print(f'  Tasa objetivo de rutas afectadas: {config["eventos"]["tasa_rutas_objetivo"]}')
    for tipo, prob in config['eventos']['probabilidad_base'].items():
        print(f'  {tipo:12s}: prob. base {prob}')
else:
    print(f'No se encontró {config_path}. Ejecuta desde la raíz del repositorio.')

## 3. Inspeccionar la red vial construida

La red se construye exclusivamente con datos reales de OpenStreetMap (Overpass API + extracto Geofabrik Perú + OSMnx). Cada nodo y tramo proviene de un objeto OSM identificado por su `id` global.

In [ ]:
# Cargar resumen de la red vial
resumen_path = os.path.join(RED_VIAL_DIR, 'resumen.json')

if os.path.exists(resumen_path):
    with open(resumen_path, 'r', encoding='utf-8') as f:
        resumen = json.load(f)
    
    print('=== Red vial construida (OpenStreetMap) ===')
    print(f'Nodos operativos:      {resumen["n_nodos"]:>10,}')
    print(f'Tramos consolidados:    {resumen["n_tramos"]:>10,}')
    print(f'Longitud total:         {resumen["km_totales"]:>10.1f} km')
    print(f'Nodos OSM procesados:   {resumen["nodos_osm_totales"]:>10,}')
    print(f'Ciudades:               {resumen["ciudades"]:>10}')
    print(f'Distribuidores:         {resumen["n_distribuidores"]:>10}')
    print(f'Grafo conexo:           {resumen["conectado"]}')
    print(f'Tramos unidireccionales:{resumen["n_unidireccionales"]:>10,}')
    print(f'Restricción de pesados: {resumen["n_restriccion_pesados"]:>10} ({resumen["km_restriccion_pesados"]} km)')
    
    print(f'\nValidación: {"OK" if resumen["validacion"]["ok"] else "FALLÓ"}')
    print(f'Completitud: {resumen["validacion"]["metricas"]["completitud_pct"]}%')
    print(f'Tramos con alterna: {resumen["validacion"]["metricas"]["tramos_con_alterna_pct"]}%')
else:
    print('No se encontró resumen.json. Ejecuta primero: python scripts/01_red_vial.py')

## 4. Cargar nodos y tramos de la red

In [ ]:
# Cargar nodos y tramos
nodos_path = os.path.join(RED_VIAL_DIR, 'nodos.csv')
tramos_path = os.path.join(RED_VIAL_DIR, 'tramos.csv')

if os.path.exists(nodos_path) and os.path.exists(tramos_path):
    nodos = pd.read_csv(nodos_path)
    tramos = pd.read_csv(tramos_path)
    
    print(f'Nodos: {len(nodos):,} filas, {nodos.shape[1]} columnas')
    print(f'Tramos: {len(tramos):,} filas, {tramos.shape[1]} columnas')
    
    print(f'\nPrimeras 5 filas de nodos:')
    print(nodos.head())
    
    print(f'\nPrimeras 5 filas de tramos:')
    print(tramos.head())
    
    # Distribución por tipo de vía
    print(f'\nDistribución de tramos por tipo de vía:')
    print(tramos['tipo_via'].value_counts().to_string())
else:
    print('No se encontraron los archivos de red. Ejecuta primero: python scripts/01_red_vial.py')

## 5. Visualizar la distribución de tramos por tipo de vía

In [ ]:
# Visualizar distribución de tipos de vía
if 'tramos' in dir() and len(tramos) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Conteo por tipo de vía
    tipo_counts = tramos['tipo_via'].value_counts()
    colors = ['#1a5276', '#2874a6', '#3498db', '#85c1e9', '#aed6f1', '#d4e6f1']
    axes[0].barh(tipo_counts.index, tipo_counts.values, color=colors[:len(tipo_counts)])
    axes[0].set_xlabel('Número de tramos')
    axes[0].set_title('Tramos por tipo de vía')
    axes[0].invert_yaxis()
    
    # Distribución de distancias
    axes[1].hist(tramos['distancia_km'], bins=80, color='#2874a6', edgecolor='white', alpha=0.8)
    axes[1].set_xlabel('Distancia (km)')
    axes[1].set_ylabel('Frecuencia')
    axes[1].set_title('Distribución de distancias por tramo')
    axes[1].set_xlim(0, 50)
    
    plt.tight_layout()
    plt.savefig('visualizacion_red_vial.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Gráfico guardado como visualizacion_red_vial.png')
else:
    print('Carga primero los tramos en la celda anterior.')

## 6. Macro-tramos del corredor

In [ ]:
# Cargar y mostrar los macro-tramos
corredor_path = os.path.join(RED_VIAL_DIR, 'corredor_macro.csv')

if os.path.exists(corredor_path):
    corredor = pd.read_csv(corredor_path)
    
    print('=== Macro-tramos del corredor ===')
    print(f'{"Tramo":15s} {"Origen":12s} {"Destino":12s} {"Km":>8s} {"Tiempo":>8s} {"#Tramos":>8s} {"Tipo":12s}')
    print('-' * 80)
    for _, row in corredor.iterrows():
        print(f'{row["id_macro"]:15s} {row["origen"]:12s} {row["destino"]:12s} '
              f'{row["distancia_km"]:>8.1f} {row["tiempo_min"]:>7.1f}h {row["n_tramos"]:>8} {row["tipo_via_dominante"]}')
    
    corredor_norte = corredor[corredor['alcance'] == 'corredor']
    extensiones = corredor[corredor['alcance'] == 'extension']
    print(f'\nTotal corredor norte: {corredor_norte["distancia_km"].sum():.1f} km ({len(corredor_norte)} tramos)')
    print(f'Total extensiones:    {extensiones["distancia_km"].sum():.1f} km ({len(extensiones)} tramos)')
else:
    print('No se encontró corredor_macro.csv.')

## 7. Benchmark con viajes reales de la empresa

Los 500 viajes registrados por la empresa se reconstruyeron sobre la red vial real. Este benchmark sirve como línea base de la operación manual para comparar contra la política MARL.

In [ ]:
# Cargar benchmark
bench_resumen_path = os.path.join(BENCHMARK_DIR, 'benchmark_resumen.json')

if os.path.exists(bench_resumen_path):
    with open(bench_resumen_path, 'r', encoding='utf-8') as f:
        bench = json.load(f)
    
    print('=== Benchmark de viajes reales ===')
    print(f'Puntos de entrega:     {bench["n_puntos"]}')
    print(f'Puntos fuera:          {bench["n_puntos_fuera"]}')
    print(f'Viajes totales:        {bench["n_viajes"]}')
    print(f'Viajes válidos:        {bench["n_viajes_validos"]}')
    print(f'Viajes con error:      {bench["n_viajes_error"]}')
    print(f'Km medio por red:      {bench["km_red_medio"]}')
    print(f'Km medio geodésico:    {bench["km_geo_medio"]}')
    print(f'Ratio red/geodésica:   {bench["ratio_red_geo"]}')
    print(f'Snap máximo:           {bench["dist_snap_max_m"]} m')
else:
    print('No se encontró el benchmark. Ejecuta primero: python scripts/06_integrar_viajes.py')

## 8. Distribución de viajes por ciudad destino

In [ ]:
# Cargar viajes del benchmark y visualizar
viajes_path = os.path.join(BENCHMARK_DIR, 'benchmark_viajes.csv')

if os.path.exists(viajes_path):
    viajes = pd.read_csv(viajes_path)
    
    print(f'Viajes cargados: {len(viajes)}')
    print(f'Columnas: {list(viajes.columns)}')
    
    # Histograma de kilómetros
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    col_km = 'km_red' if 'km_red' in viajes.columns else viajes.columns[3]
    axes[0].hist(viajes[col_km], bins=40, color='#1a5276', edgecolor='white', alpha=0.8)
    axes[0].set_xlabel('Kilómetros por red')
    axes[0].set_ylabel('Número de viajes')
    axes[0].set_title(f'Distribución de kilómetros por viaje (n={len(viajes)})')
    axes[0].axvline(viajes[col_km].mean(), color='red', linestyle='--', label=f'Media: {viajes[col_km].mean():.0f} km')
    axes[0].legend()
    
    # Duración
    col_min = 'min_red' if 'min_red' in viajes.columns else None
    if col_min:
        axes[1].hist(viajes[col_min] / 60, bins=40, color='#2874a6', edgecolor='white', alpha=0.8)
        axes[1].set_xlabel('Duración (horas)')
        axes[1].set_ylabel('Número de viajes')
        axes[1].set_title('Distribución de duración por viaje')
    
    plt.tight_layout()
    plt.savefig('benchmark_viajes.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Gráfico guardado como benchmark_viajes.png')
else:
    print('No se encontró benchmark_viajes.csv.')

## 9. Rutas de referencia (Valhalla con perfil camión)

Valhalla calcula rutas con perfil de camión real (hgv, maxweight, maxheight). Se usa fuera del bucle de aprendizaje como referencia externa.

In [ ]:
# Cargar rutas de Valhalla
rutas_path = os.path.join(VALHALLA_DIR, 'rutas_od.csv')

if os.path.exists(rutas_path):
    rutas = pd.read_csv(rutas_path)
    
    print('=== Rutas de referencia (Valhalla) ===')
    print(f'{"Par":12s} {"Motor (km)":>10s} {"Red propia":>10s} {"Desvío":>8s} {"Auto (km)":>10s}')
    print('-' * 55)
    for _, row in rutas.iterrows():
        print(f'{row["id_macro"]:12s} {row["km_valhalla"]:>10.1f} {row["km_red_propia"]:>10.1f} '
              f'{row["desvio_pct"]:>7.2f}% {row["km_auto"]:>10.1f}')
    
    print(f'\nDesvío medio vs auto: {rutas["desvio_auto_pct"].mean():.2f}%')
    print(f'Desvío medio vs truck: {rutas["desvio_pct"].mean():.2f}%')
else:
    print('No se encontró rutas_od.csv. Ejecuta primero: python scripts/08_rutas_valhalla.py')

## 10. Modelo Dec-POMDP — Espacio de acciones

El simulador modela el problema como un Dec-POMDP cooperativo con 4 acciones discretas por agente:

In [ ]:
# Descripción del modelo
acciones = {
    0: {'nombre': 'continuar', 'descripcion': 'Seguir el plan vigente sin recalcular'},
    1: {'nombre': 'tomar_alterna', 'descripcion': 'Rodear el tramo planificado (desvío real, +2% km)'},
    2: {'nombre': 'esperar', 'descripcion': 'Detenerse un paso (15 min), costo S/ 64/h'},
    3: {'nombre': 'replanificar', 'descripcion': 'Recalcular ruta con Dijkstra ponderado por eventos'},
}

print('=== Espacio de acciones del Dec-POMDP ===')
print()
for codigo, info in acciones.items():
    print(f'  {codigo} — {info["nombre"]:15s}: {info["descripcion"]}')

print()
print('Configuración del entorno (config/entrenamiento.yaml):')
print('  Agentes por episodio:   3')
print('  Horizonte:              240 pasos (60 horas)')
print('  Duración del paso:      15 min')
print('  Observación local:      20 dimensiones')
print('  Estado global (CTDE):   58 dimensiones')
print('  Descuento (γ):          0.99')
print('  Descanso conductor:     45 min cada 5 h')
print('  Max replanificaciones:  12 por agente')

## 11. Logs preliminares de entrenamiento

Se ejecutaron pruebas preliminares de QMIX y MAPPO. Los logs muestran que el pipeline funciona y las curvas de recompensa muestran tendencia de mejora.

In [ ]:
# Cargar y visualizar logs de entrenamiento
qmix_log_path = os.path.join(ARTIFACTS_DIR, 'logs', 'qmix_semilla1.csv')
mappo_log_path = os.path.join(ARTIFACTS_DIR, 'logs', 'mappo_semilla1.csv')

logs_cargados = False

if os.path.exists(qmix_log_path):
    qmix_log = pd.read_csv(qmix_log_path)
    # Tomar los primeros 200 episodios (primera corrida)
    qmix_log = qmix_log.head(200)
    logs_cargados = True
    print(f'QMIX: {len(qmix_log)} episodios cargados')
else:
    print('No se encontró log de QMIX.')

if os.path.exists(mappo_log_path):
    mappo_log = pd.read_csv(mappo_log_path)
    print(f'MAPPO: {len(mappo_log)} episodios cargados')
else:
    print('No se encontró log de MAPPO.')

if logs_cargados:
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # Recompensa QMIX
    axes[0, 0].plot(qmix_log['episodio'], qmix_log['recompensa'], alpha=0.3, color='#2874a6', label='Por episodio')
    window = 20
    if len(qmix_log) > window:
        rolling = qmix_log['recompensa'].rolling(window=window).mean()
        axes[0, 0].plot(qmix_log['episodio'], rolling, color='#1a5276', linewidth=2, label=f'Media móvil ({window})')
    axes[0, 0].set_xlabel('Episodio')
    axes[0, 0].set_ylabel('Recompensa')
    axes[0, 0].set_title('QMIX — Curva de recompensa (semilla 1)')
    axes[0, 0].legend()
    axes[0, 0].axhline(0, color='gray', linestyle=':', alpha=0.5)
    
    # Puntualidad QMIX
    axes[0, 1].plot(qmix_log['episodio'], qmix_log['puntualidad_pct'], alpha=0.4, color='#2874a6')
    if len(qmix_log) > window:
        rolling_pct = qmix_log['puntualidad_pct'].rolling(window=window).mean()
        axes[0, 1].plot(qmix_log['episodio'], rolling_pct, color='#1a5276', linewidth=2)
    axes[0, 1].set_xlabel('Episodio')
    axes[0, 1].set_ylabel('Puntualidad (%)')
    axes[0, 1].set_title('QMIX — Puntualidad')
    axes[0, 1].set_ylim(-5, 105)
    
    # Recompensa MAPPO
    if 'mappo_log' in dir():
        axes[1, 0].plot(mappo_log['episodio'], mappo_log['recompensa'], alpha=0.5, color='#c0392b', marker='o', markersize=3)
        axes[1, 0].set_xlabel('Episodio')
        axes[1, 0].set_ylabel('Recompensa')
        axes[1, 0].set_title('MAPPO — Curva de recompensa (semilla 1)')
        axes[1, 0].axhline(0, color='gray', linestyle=':', alpha=0.5)
        
        axes[1, 1].plot(mappo_log['episodio'], mappo_log['puntualidad_pct'], alpha=0.5, color='#c0392b', marker='o', markersize=3)
        axes[1, 1].set_xlabel('Episodio')
        axes[1, 1].set_ylabel('Puntualidad (%)')
        axes[1, 1].set_title('MAPPO — Puntualidad')
        axes[1, 1].set_ylim(-5, 105)
    
    plt.tight_layout()
    plt.savefig('curvas_entrenamiento.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Gráfico guardado como curvas_entrenamiento.png')

## 12. Diagnóstico de movimiento: verificación del simulador

El script `05_diagnostico_movimiento.py` ejecuta un episodio con la política trivial «continuar» y verifica que los vehículos recorren la topología OSM real.

In [ ]:
# Resumen del diagnóstico de movimiento
print('=== Diagnóstico de movimiento (política «continuar») ===')
print()
print('Red cargada:           18 167 nodos, 28 642 tramos')
print('Episodio:              3 agentes, 240 pasos de 15 min (60 h)')
print('Km recorridos:         986.8 / 994.9 / 563.1 km por agente')
print('Entregas completadas:  6 de 6, sin pendientes')
print('Planes generados:      1 por agente (506, 499 y 278 nodos)')
print('Esperas/replanif.:     0 / 0 con política «continuar»')
print('Descanso de jornada:   45 min cada 5 h de conducción')
print()
print('--- Resultado por acción ---')
print('Acción 0 (continuar):     986.8 / 994.9 / 563.1 km | 6/6 entregas')
print('Acción 1 (tomar alterna): 1007.0 / 1024.1 / 582.3 km (+2.0%) | 6/6 entregas')
print('Acción 2 (esperar):       9.9 / 9.9 / 9.9 km (239 esperas) | 0/6 entregas')
print('Acción 3 (replanificar):  986.8 / 994.9 / 563.1 km (12 replanif.) | 6/6 entregas')
print()
print('Conclusión: las 4 acciones producen resultados distintos y coherentes.')
print('La acción «tomar alterna» no es muerta: encarece la ruta 2% real.')

## 13. Matriz de decisión del algoritmo MARL

Se evaluaron QMIX, MAPPO, MADDPG y VDN con cinco criterios ponderados.

In [ ]:
# Matriz de decisión
criterios = ['C1. Acción discreta (25%)', 'C2. Eficiencia muestral (25%)',
             'C3. Asignación de crédito (20%)', 'C4. Estabilidad (15%)',
             'C5. Implementación (15%)']
algoritmos = ['QMIX', 'MAPPO', 'MADDPG', 'VDN']
puntajes = {
    'QMIX':   [5, 5, 4, 4, 3],
    'MAPPO':  [5, 2, 4, 3, 4],
    'MADDPG': [2, 3, 4, 2, 2],
    'VDN':    [5, 5, 2, 3, 5],
}
pesos = [0.25, 0.25, 0.20, 0.15, 0.15]

print('=== Matriz de decisión del algoritmo MARL ===')
print()
header = f'{"Criterio":35s}' + ''.join(f'{a:>8s}' for a in algoritmos)
print(header)
print('-' * len(header))

for i, crit in enumerate(criterios):
    row = f'{crit:35s}'
    for alg in algoritmos:
        row += f'{puntajes[alg][i]:>8d}'
    print(row)

print('-' * len(header))
row = f'{"Puntaje ponderado":35s}'
for alg in algoritmos:
    total = sum(p * w for p, w in zip(puntajes[alg], pesos))
    row += f'{total:>8.2f}'
print(row)

print()
print('Decisión: QMIX (principal) + MAPPO (contraste)')
print('MADDPG descartado: diseñado para acciones continuas')
print('VDN descartado: no captura interacciones no lineales')

## 14. Función de recompensa

La recompensa cooperativa es el promedio de las recompensas individuales, alineando los incentivos (equipo, no competencia).

In [ ]:
# Función de recompensa
print('=== Función de recompensa ===')
print()
print('r_t = -w_c * (Δcosto_t / 500) - w_t * (15 / 600) - penal_retraso - w_d * (0.002) + φ_entrega')
print()
print('Términos:')
print('  w_c (costo incremental):   0.60 | normalizado por S/ 500')
print('  w_t (tiempo):              0.40 | normalizado por 600 min')
print('  w_d (desvío):              0.30 | penalización por tramo')
print('  φ_entrega:                 +5.0 si cumple ventana, -3.0 si no')
print('  Penalidad por bloqueo:     1.50 | 0.05 por paso bloqueado')
print('  Penalidad por replanificar:0.10 | 0.01 por evento')
print()
print('Recompensa cooperativa: promedio de recompensas individuales')
print('Se usan incrementos (no valores acumulados) para evitar sesgo hacia episodios cortos.')

## 15. Próximos pasos

| Paso | Descripción | Dependencia |
|---|---|---|
| 1 | Recibir direcciones reales de los 17 distribuidores | Wilson |
| 2 | Recibir Excel de entidades (recursos y operación) | Wilson |
| 3 | Sustituir datos sintéticos por reales | Franklin |
| 4 | Entrenar QMIX: 2000 episodios × 5 semillas | Franklin |
| 5 | Entrenar MAPPO como contraste | Franklin |
| 6 | Evaluación estadística: Mann-Whitney vs. 3 baselines | Franklin |
| 7 | Preparar presentación semana 8 (examen parcial) | Equipo |

---

### Comandos para ejecutar el pipeline completo

```bash
# Red vial
python scripts/01_red_vial.py

# Dataset
python scripts/02_dataset.py --viajes 180

# Diagnóstico
python scripts/05_diagnostico_movimiento.py

# Benchmark
python scripts/06_integrar_viajes.py
python scripts/07_cobertura_muestra.py

# Rutas de referencia (requiere Valhalla)
python scripts/08_rutas_valhalla.py
python scripts/09_alternativas.py

# Entrenamiento
python scripts/03_entrenar.py --algoritmo qmix --episodios 2000 --semillas 1 2 3

# Evaluación
python scripts/04_evaluar.py --modelo artifacts/modelos/qmix_s1.pt --episodios 30
```

---

**Repositorio:** https://github.com/fbarrioss/trabajo-investigacion-ii.git  
**Semilla:** 20261016